# 가맹점 벡터 만들기 (Colab GPU) — 데이터 버전 1

로컬 `load-prepare`가 만든 `embed_input.csv`(store_id, doc_text)를 서비스 임베딩 모델로 벡터로 만듭니다. 사용법 전체는 `docs/dev/DATA_LOAD.md`.

**Drive에 미리 둘 것** (`내 드라이브/store-search-ai/`)

| 경로 | 내용 |
|---|---|
| `runs/finetune/bge_m3_ft_mj_20261014_0743/` | 서비스 모델(이미 있음) |
| `service_load/v1/embed_input.csv` | 로컬 `data/load/embed_input.csv`를 그대로 올림 |

**결과** (같은 `service_load/v1/` 폴더): `embeddings.npy`(float32 [가게 수, 1024], 입력과 같은 순서), `embeddings.json`(모델·건수·입력 파일 sha256). 둘 다 로컬 `data/load/`에 내려받아 `load-import`를 실행합니다.

**설정은 임베딩 서버(`embedding-server/app/main.py`)와 같아야 합니다**: fp32, max_seq_length 128, 정규화. 서비스 평가는 fp16으로 했지만, 저장 벡터는 검색 때 CPU 서버(fp32)가 만드는 질의 벡터와 맞추려고 fp32로 만듭니다. `load-import`가 표본 32개를 로컬 서버로 다시 계산해 코사인 0.999 이상인지 확인합니다.

런타임 → 런타임 유형 변경 → GPU(T4 가능) → 위에서부터 실행. T4에서 22만 개 약 10~15분.

In [ ]:
# 임베딩 서버와 같은 버전(embedding-server/requirements.txt). torch는 Colab 것을 그대로 쓴다
!pip install -q sentence-transformers==5.7.0 transformers==5.18.0

In [ ]:
import torch
assert torch.cuda.is_available(), "런타임 > 런타임 유형 변경 > GPU를 선택하세요"
print(torch.cuda.get_device_name(0), torch.__version__)
# fp32 그대로 계산(TF32로 줄이지 않음) — CPU 서버 결과와 맞추려고
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd

ROOT = Path("/content/drive/MyDrive/store-search-ai")
MODEL_DIR = ROOT / "runs/finetune/bge_m3_ft_mj_20261014_0743"
LOAD_DIR = ROOT / "service_load/v1"
INPUT = LOAD_DIR / "embed_input.csv"
MAX_SEQ_LENGTH = 128   # 임베딩 서버 MAX_SEQ_LENGTH와 같게
BATCH = 256

input_sha256 = hashlib.sha256(INPUT.read_bytes()).hexdigest()
# 문자열 그대로 읽는다("nan"·"NULL" 같은 상호를 빈 값으로 바꾸지 않게)
docs = pd.read_csv(INPUT, dtype=str, keep_default_na=False, encoding="utf-8")
assert list(docs.columns) == ["store_id", "doc_text"], docs.columns
assert docs["store_id"].is_unique and (docs["doc_text"].str.len() > 0).all()

manifest_path = MODEL_DIR / "model_manifest.json"
MODEL_TAG = json.loads(manifest_path.read_text(encoding="utf-8"))["tag"] if manifest_path.exists() else MODEL_DIR.name
print(f"가게 {len(docs):,}개, 입력 sha256 {input_sha256[:12]}…, 모델 {MODEL_TAG}")
print("가장 긴 원문", docs["doc_text"].str.len().max(), "자")
docs.head(3)

In [ ]:
import time

import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(str(MODEL_DIR), device="cuda", model_kwargs={"dtype": torch.float32})
model.max_seq_length = MAX_SEQ_LENGTH
print(model)

started = time.time()
vectors = model.encode(docs["doc_text"].tolist(), batch_size=BATCH, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
print(f"{time.time() - started:.0f}초", vectors.shape, vectors.dtype)

In [ ]:
norms = np.linalg.norm(vectors, axis=1)
assert vectors.shape == (len(docs), 1024), vectors.shape
assert np.isfinite(vectors).all()
assert np.abs(norms - 1).max() < 1e-3, np.abs(norms - 1).max()

# 순서 확인: 한 개씩 다시 계산해도 같은 벡터
for i in [0, len(docs) // 2, len(docs) - 1]:
    one = model.encode([docs["doc_text"].iloc[i]], normalize_embeddings=True, convert_to_numpy=True)[0]
    cos = float(one @ vectors[i])
    print(i, docs["doc_text"].iloc[i][:40], round(cos, 6))
    assert cos > 0.9999

In [ ]:
import shutil
from datetime import datetime, timezone

import sentence_transformers

# 로컬 디스크에 먼저 쓰고 Drive로 복사(쓰다 끊겨 반쪽 파일이 Drive에 남지 않게)
local = Path("/content/out")
local.mkdir(exist_ok=True)
np.save(local / "embeddings.npy", np.ascontiguousarray(vectors, dtype="<f4"))
manifest = {
    "model": MODEL_TAG,
    "count": int(len(docs)),
    "dimension": int(vectors.shape[1]),
    "dtype": "float32",
    "normalized": True,
    "max_seq_length": MAX_SEQ_LENGTH,
    "input_sha256": input_sha256,
    "device": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "created_at": datetime.now(timezone.utc).isoformat(),
}
(local / "embeddings.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
for name in ["embeddings.npy", "embeddings.json"]:
    shutil.copy(local / name, LOAD_DIR / name)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Drive:", LOAD_DIR, f"embeddings.npy {(LOAD_DIR / 'embeddings.npy').stat().st_size / 1e6:.0f}MB")